In [1]:
%%capture
from pyfiles.preamble import *
setup_notebook()

get_ipython().run_line_magic('run', '1_create_scenario.ipynb')

# Optimiser — finding endogenous capacities

This notebook minimises **total annual system costs** over a set of user-defined capacity variables using Nelder-Mead optimisation. It builds directly on the scenario parameters defined in `1_create_scenario.ipynb`.

**Workflow**
1. Pick the capacity variables to optimise (`x0`) and set starting values
2. Optionally constrain variables with lower/upper bounds
3. Set a maximum electricity import and a run budget (`max_evaluations`)
4. Run the solver for each scenario — results summarised at the bottom

Browse `pyfiles/input_variables.py` for available parameter names.

### 1. Choice variables

Each entry is `'parameter_name': initial_value`. The solver will optimise over all listed variables simultaneously. Browse `pyfiles/input_variables.py` for available names.

In [ ]:
x0 = {
    'input_cap_pp2_el'          : 4_500.0,   # MW
    'input_cap_ELTtrans_el'     : 4_000.0,   # MW
    'input_H2storage_trans_cap' : 50.0,      # MW
}

### 2. Bounds *(optional)*

Constrain any choice variable with `(lower, upper)` — use `None` for no bound on that side. Variables not listed here are unconstrained.

In [3]:
bounds = {
    'input_cap_ELTtrans_el'     : (3200.0, None),
    'input_H2storage_trans_cap' : (32.0,   None),
}

### 3. Solver settings

- **`import_limit_twh`** — maximum annual electricity imports allowed (TWh); violations are penalised. Set to `0` to require self-sufficiency.
- **`max_evaluations`** — budget of EnergyPLAN calls. More complex problems (more variables, tighter bounds) need more evaluations to converge; start low to test, then increase.

In [4]:
import_limit_twh  = 0.5    # TWh — electricity import constraint
max_evaluations   = 3

### 4. Run

In [5]:
# base
solver.configure(
    'base', base_params, base_case_params, shock_case_params,
    x0=x0, bounds=bounds, ref_file=ref_path,
    import_limit_twh=import_limit_twh,
)
out_base = solver.run(max_evaluations=max_evaluations)

# shock
solver.configure(
    'shock', base_params, base_case_params, shock_case_params,
    x0=x0, bounds=bounds, ref_file=ref_path,
    import_limit_twh=import_limit_twh,
)
out_shock = solver.run(max_evaluations=max_evaluations)

EnergyPLAN cost minimiser  (case='base', 3 max evals)
  Choice vars (3): cap_pp2_el=4500,  cap_ELTtrans_e=4000,  H2storage_tran=50
  Import limit : 0.5 TWh
  Bound: input_cap_ELTtrans_el  >= 3200.0
  Bound: input_H2storage_trans_cap  >= 32.0
 *[  1] cap_pp2_el= 4500.0  cap_ELTtrans_e= 4000.0  H2storage_tran=   50.0  → cost=23082.00 M EUR  imports=0.140 TWh  (17.3s)
  [  2] cap_pp2_el= 4725.0  cap_ELTtrans_e= 4000.0  H2storage_tran=   50.0  → cost=23089.00 M EUR  imports=0.110 TWh  (16.9s)
 *[  3] cap_pp2_el= 4500.0  cap_ELTtrans_e= 4200.0  H2storage_tran=   50.0  → cost=23073.00 M EUR  imports=0.120 TWh  (16.5s)

Finished after 3 evaluations  (converged: False)
  Best cost : 23073.00 M EUR
  Best parameters:
    input_cap_pp2_el                    = 4500.0
    input_cap_ELTtrans_el               = 4200.0
    input_H2storage_trans_cap           = 50.0
EnergyPLAN cost minimiser  (case='shock', 3 max evals)
  Choice vars (3): cap_pp2_el=4500,  cap_ELTtrans_e=4000,  H2storage_tran=50
  Imp

### 5. Results

In [6]:
results = {'Base': out_base, 'Shock': out_shock}

rows = []
for name, out in results.items():
    row = {'Scenario': name, 'Total annual costs (M EUR)': round(out['best_cost'], 1)}
    for param, val in zip(x0.keys(), out['best_x']):
        from pyfiles.input_variables import labels as _inp_labels
        col = _inp_labels.get(param, param)
        row[col] = round(val, 1)
    rows.append(row)

pd.DataFrame(rows).set_index('Scenario')

,Total annual costs (M EUR),Peak load plant (PP2) capacity (MW),Electrolysis (transport) capacity (MW),H2 storage transport capacity (TWh)
Scenario,,,,
Base,23073.0,4500.0,4200.0,50.0
Shock,23392.0,4500.0,4200.0,50.0
